# Automatic Deep Research 

Welcome to this new practice lab! By now you should have a clearer view of the elements that compose a multi-agent system. In this lab you will get to put it into action by creating your first crew.

**What you'll learn:**
- How to define agents with specific roles and expertise
- How to provide agents with tools to perform their tasks
- How to create your own tasks that agents will execute
- How to assemble agents and tasks into a Crew, all using CrewAI

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an automatic deep research solution that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter. 

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

If you are stuck, or simply want to copy a solution into your notebook so that you can execute it, you can find all solution code inside the [Solution](Solution) folder.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Understanding the problem](#1)
- [2. Set up your notebook](#2)
- [3. Define the Agents](#3)
  - [3.1. Create tool instances](#3-1)
  - [3.2. Define the Research Planner agent](#3-2)
  - [3.3. Define the remaining agents](#3-3)
- [4. Create the Tasks](#4)
  - [4.1. Define the Create research plan task](#4-1)
  - [4.2. Define the remaining tasks](#4-2)
- [5. Define the Crew and get the results](#5)

<a id="1"></a>

## 1. Understanding the problem
In this lab, you will focus on building a custom deep research crew. This Crew will be in charge of creating a research plan based on the user's input, and executing it, while reviewing and checking the facts. Finally, with the gathered information a report needs to be generated.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task? 

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<img src="/Users/bohaoli/Desktop/tuto/tuto_langchain/official/langchain-langgraph-langsmith/deep-learning-ai/crew_ai/m1_Foundation_of_AI_Agents/06_multi-agent-systems/images/agents-tasks-diagram.png">

<a id="2"></a>

## 2. Set up your notebook

Before you start coding, run the next two cells to import all necessary modules and configure the environment variables. 

In [2]:
from crewai import Agent, Task, Crew
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

<a id="3"></a>

## 3. Define the Agents

Based on the diagram, you should have four agents:
- **Research Planner**: its goal is to analyze queries and break them down into smaller, specific research topics.
- **Internet Researcher**: its job is to perform research tasks.
- **Fact checker**: its goal is to review information for fact accuracy to avoid misinformation. 
- **Report Writer**: is in charge of writing reports, based on gathered information.

<a id="3-1"></a>

### 3.1. Create tool instances
As you can see in the diagram, you will be providing the **Internet Researcher Agent** with tools, so that it can better do their job. In particular, you will give this agent access to search the internet and scrape information from the retrieved webpages. 

There are different tools inside CrewAI you can use to search the web, in this lab you will use the [**EXA Search Web Loader**](https://docs.crewai.com/en/tools/search-research/exasearchtool#exa-search-web-loader) tool, which is designed to perform a semantic search for a specified query from a text’s content across the internet. It utilizes the [exa.ai](https://exa.ai/) API to fetch and display the most relevant search results based on the query provided by the user. exa.ai enhances semantic search by capturing richer contextual relationships between concepts, allowing for more precise information retrieval than conventional embedding approaches.

For webscraping, you will use the [**Scrape Website**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool) tool, which is designed to extract and read the content of a specified website.

In the next cell you will define instances of these tools, so you can later assign them to the agents.

In [4]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from utils import get_exa_api_key

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

### START CODE HERE ###

# Create the EXASearchTool instance
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
# Create the ScrapeWebsiteTool instance
scrape_website_tool = ScrapeWebsiteTool()

### END CODE HERE ###

TypeError: str expected, not NoneType

<a id="3-2"></a>

### 3.2. Define the Research Planner agent

In the cell below, you will see how you can create the first agent. This time, all the parameters are set up for you. Here is a quick recap of what each of the parameters represent:

- `Role`: If this was a person doing the job, what title would they have?
- `Goal`: What is the goal this agent in particular is trying to accomplish? Make sure to write concrete goal
- `Background`: it should be something the highlights the skills of the agent relevant to its role. Make sure to use keywords that will actually help your agent get better results.

In the labs, we have added two parameters not shown in the demo videos: `max_rpm`, and `max_iter`. `max_rpm` sets the maximum requests per minute to avoid rate limits, while `max_iter` limits the maximum iterations before the agent must provide its best answer. Setting these two parameters helps make the agents run a little faster, so the lab doesn't take as long to complete. 

In [16]:
# define the research planner agent
research_planner = Agent(
    role="Research Planner",
    goal="Analyze queries and break them down into smaller, specific research topics.",
    backstory=(
         "You are a research strategist who excels at breaking down complex questions "
         "into manageable research components. You identify what needs to be researched "
         "and create clear research objectives."
    ),
    verbose=True, # set to True to see detailed agent actions
    max_rpm=150,
    max_iter=15
)

<a id="3-3"></a>

### 3.3. Define the remaining agents

Now you can define the three remaining agents. The `role` and `goal` parameters are already filled in for you; use your own creativity to fill in the `backstory`.  

Do not forget to assign the tools to the **Internet Researcher** and **Fact Checker** agents. You can do this by setting the `tools` argument.

In [17]:

researcher = Agent(
    role="Internet Researcher",
    goal="Research thoroughly all assigned topics",
    ### START CODE HERE ###
    backstory=(
        "You are a skilled researcher with experience in online investigation "
        "and data collection. You know how to find reliable sources, extract relevant information, "
        "and always verify facts across multiple sources to avoid misinformation or hallucination. "
        "You never invent facts and always trace information to its origin."
    ),
    # add the 2 tool instances you created
    tools=[exa_search_tool, scrape_website_tool],
    ### END CODE HERE ###
    verbose=True,
    max_rpm=150,
    max_iter=15
)

fact_checker = Agent(
    role="Fact Checker",
    goal=(
        "Verify data for accuracy, identify inconsistencies, "
        "and flag potential misinformation"
    ),
    ### START CODE HERE ###
    backstory=(
        "You are a quality assurance specialist with expertise in fact-checking "
        "and identifying misinformation and hallucinations. You cross-reference information, "
        "spot inconsistencies, and ensure all data meets high accuracy standards. You rigorously"
        "check for hallucinated or invented content and require that all facts be supported"
        "by evidence."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    ### END CODE HERE ###
    verbose=True,
    max_rpm=150,
    max_iter=15
)

report_writer = Agent(
    role="Report Writer",
    goal="Write clear, concise, and well-structured reports based on gathered information",
    ### START CODE HERE ###
    backstory=(
         "You are an expert writer who specializes in creating clear, well-structured "
        "research reports. You synthesize complex information into readable formats and "
        "always include proper citations and sources."
    ),
    ### END CODE HERE ###
    verbose=True,max_rpm=150,
    max_iter=15
)
        

<a id="4"></a>

## 4. Create the Tasks

Now that you have set up the agents, it is time to define the tasks. If you go back to the diagram, you will see you need four tasks:

- **Create research plan**: Based on the user's query, break it down into specific topics and key questions, and create a focused research plan.
    - Output: A research plan with main research topics to investigate, key questions for each topic, and success criteria for the research.

- **Gather research data**: Using the research plan, collect information on all identified topics. Cite all sources used.
    - Output: Comprehensive research data including: information for each research topic, and citations used along with source credibility notes.

- **Verify information quality**: Review all collected research. Identify any conflicting information, potential misinformation, or gaps that need addressing.
    - Output: A report with the all the collected data, and its review. It should include consistency check results and source reliability ratings

- **Write final report**: Create a comprehensive report that answers the original query using all verified research data. Structure it with clear sections, include citations, and provide actionable insights.
    - Output: The final research report. In addition to the full answer, it should have an executive summary, and complete source citations.


For each `Task` you need to define the following parameters:
- `description`: A thorough description of the task. You can even break it down into different items.
- `expected_output`: what should the output return. Be specific, specially if you want any structure in your result, like a dictionary with specific keys.
- `agent`: who is performing the task? You need to match the task to one of the agents you already defined

In the description you will need to pass the inputs to the tasks. In this lab, you will only have as input the user's query, which will be saved as `user_query`:


<a id="4-1"></a>

### 4.1. Define the Create research plan task

In the cell below, you will see how you can create the first task. This time, all the parameters are set up for you. Notice how the context variables are passed the the description between curly brackets. 

In [18]:
# define the create research plan task
create_research_plan_task = Task(
    description=(
        "Based on the user's query, break it down into specific topics and key questions, "
        "and create a focused research plan."
        "The user's query is: {user_query}"
    ),
    expected_output=(
        "A research plan with main research topics to investigate, "
        "key questions for each topic, and success criteria for the research."
        ),
    agent=research_planner,
)

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**📝 我的笔记 — `{user_query}` 不是 f-string 插值，那它怎么被解释的？**

上面 `description` 用的是普通字符串（`"..."`），不是 `f"..."`，所以 Python 在这一步**什么都不做**，`{user_query}` 只是原样存进 `task.description` 里的字面文本。

真正做替换的是 **CrewAI 自己**，在你后面调用：

```python
result = crew.kickoff(inputs={"user_query": user_query})
```

的时候：CrewAI 内部（`Task.interpolate_inputs`）会遍历每个 task 的 `description` / `expected_output`（以及 agent 的 `role`/`goal`/`backstory`），做一次类似 `str.format(**inputs)` 的手动字符串替换，把 `{user_query}` 换成 `inputs["user_query"]` 的实际值。

- 占位符语法长得像 Python 的 `str.format()`，但替换动作是 CrewAI 库代码手写的模板插值，**不是**解释器自动做的。
- 这样设计是因为定义 `Task(...)` 的时候，`user_query` 这个变量可能还没被赋值——CrewAI 想要"先定义任务模板，运行时再灌数据"，跟 LangChain 的 `PromptTemplate` 思路一样。

</div>


<a id="4-2"></a>

### 4.2. Define the remaining tasks

Now define the three remaining tasks. The `description` is already filled in for you, you will need to define the `expected_output` and `agent` for each of the Tasks.

In [19]:
# define the gather research data task
gather_research_data_task = Task(
    description=(
        "Using the research plan, collect information on all identified topics. "
        "Cite all sources used."
    ),
    ### START CODE HERE ###
    expected_output=(
        "Comprehensive research data including: information for each"
        "research topic, and citations used along with source credibility notes"
    ),
    agent=researcher,
    ### END CODE HERE ###
)

#define the verify information quality task
verify_information_quality_task = Task(
    description=(
        "Review all collected research. Identify any conflicting information, "
        "potential misinformation, or gaps that need addressing."
    ),
    ### START CODE HERE ###
    expected_output=(
        "A report with all the original data you got plus any "
        "verified facts vs. questionable information, make sure this is as comprehensive "
        "as possible for final report generation"
    ),
    agent=fact_checker,
    ### END CODE HERE ###
)

# define the write final report task
write_final_report_task = Task(
    description=(
        "Create a comprehensive report that answers the original query using all verified research data. "
        "Structure it with clear sections, include citations, and provide actionable insights."
    ),
    ### START CODE HERE ###
    expected_output=(
        "A final research report containing: executive summary, detailed "
        "findings that answer the user query, supporting evidence and analysis, complete "
        "source citations."
    ),
    agent=report_writer,
    ### END CODE HERE ###
)
    

<a id="5"></a>

## 5. Define the Crew and get the results

Once the agents and tasks have been defined, you are ready to create the crew. In order to so, you will need to set the following arguments:
- `agents`: list of agents in the crew
- `tasks`: list of tasks in the crew. The tasks should be listed in the order they should be executed

In the next cell, fill in the agents and tasks for the crew.

In [20]:
# create the crew with the defined agents and tasks
crew = Crew(
    ### START CODE HERE ###
    agents=[research_planner, researcher, fact_checker, report_writer],
    tasks=[create_research_plan_task, gather_research_data_task, verify_information_quality_task, write_final_report_task],
    ### 
)

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**📝 我的笔记 — `tasks` 是个 list，上一个 task 的 output 怎么"喂"给下一个？**

CrewAI 在 `Process.sequential`（默认模式）下按 `tasks` 列表顺序执行，每跑完一个就把它的 `TaskOutput` 存进一个内部列表 `task_outputs`。跑下一个 task **之前**，会先算这个 task 该拿到什么 context：

```python
context = self._get_context(task, task_outputs)  # crew.py 内部逻辑
```

- 如果 `Task(...)` 没写 `context=`（默认，也就是本 lab 所有 task 的写法）→ 拿到**目前为止所有前面 task** 的 `.raw` 输出，拼接成一段文本：
  ```python
  "\n\n----------\n\n".join(output.raw for output in task_outputs)
  ```
- 如果显式写 `context=[task_a, task_b]` → 只拼接指定的那几个 task 的 `.raw`。
- 如果显式写 `context=[]` → 不给任何上文，空字符串。

这段 context 字符串最终会被拼进下一个 task 的 prompt 里（`agent/utils.py::format_task_with_context`），模板大概是：

```
<task 自己的 description + expected_output>

This is the context you're working with:
<前面 task 的 raw 输出拼接文本>
```

**关键点**：这不是什么向量检索/结构化传参，纯粹是**把前一个 task 的文本输出，原样拼进下一个 task 的 prompt 里，让下一个 agent 的 LLM "读"到**。所以 `gather_research_data_task` 能看到 `create_research_plan_task` 的研究计划，靠的就是这套自动拼接，不需要你手写任何传参代码——`agents=[...]` 这个 list 本身跟这套传递机制无关，它只是团队名单/hierarchical 模式下的调度池。

</div>


Before running the crew, you need to define the query, which will be used as input for the tasks.

In [21]:
### START CODE HERE ###

# Write your query, which will be used as input for the tasks.
user_query = "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."

### END CODE HERE ###

Now you are only left with kickstarting the crew to get the results. Since you set `verbose=True` in the agents, you should monitor all the process.

In [22]:
result = crew.kickoff(
    inputs={
        "user_query": user_query,
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Based on the user's query, break it down into specific topics and key questions, and create a focused    │
│  research plan.The user's query is: Evaluate the top five emerging AI tools for automating competitive market   │
│  analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan for Evaluating the Top Five Emerging AI Tools for Automating Competitive Market Analysis**     │
│                                                                                                                 │
│  **Main Research Topics:**                                                                                      │
│                                                                                                                 │
│  1. **Identification of Top Five Emerging AI Tools**                                                            │
│     - Key Questions:                                                                                            │
│       - What are the latest AI tools specifically designed for competitive market analysis?                     │
│       - Who are the companies developing these tools?                                                           │
│       - What are the criteria for determining their emergence in the market?                                    │
│     - Success Criteria: List and briefly describe the top five AI tools identified, including the rationale     │
│  for their selection based on market trends and emergence.                                                      │
│                                                                                                                 │
│  2. **Features of Each AI Tool**                                                                                │
│     - Key Questions:                                                                                            │
│       - What specific features does each tool offer for automating competitive market analysis?                 │
│       - How do these features enhance the process of competitive analysis?                                      │
│       - Are there any unique capabilities that set these tools apart from traditional methods?                  │
│     - Success Criteria: Provide a detailed comparison table highlighting features such as data integration,     │
│  analysis capabilities, reporting tools, and user interface design.                                             │
│                                                                                                                 │
│  3. **Limitations of Each AI Tool**                                                                             │
│     - Key Questions:                                                                                            │
│       - What are the reported limitations or challenges associated with each AI tool?                           │
│       - How do these limitations affect usability in a mid-sized marketing firm?                                │
│       - Are there any known issues or customer feedback that highlight these limitations?                       │
│     - Success Criteria: Document the limitations of each tool, along with any case studies or user              │
│  testimonials that illustrate these challenges.                                                                 │
│                                                                                                                 │
│  4. **Cost Analysis of Each AI Tool**                                                                           │
│     - Key Questions:                                   

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, collect information on all identified topics. Cite all sources used.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Thought: I need to start by identifying the top five emerging AI tools specifically designed for competitive   │
│  market analysis. I will search for recent information on this topic to find credible sources and list those    │
│  tools along with their developers and emergence criteria.                                                      │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "top emerging AI tools for competitive market analysis 2024",                                │
│    "start_published_date": null,                                                                                │
│    "end_published_date": null,                                                                                  │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "website_url": "https://energent.ai/use-cases/en/industry-analysis"                                          │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  The following text is scraped website content:                                                                 │
│                                                                                                                 │
│  Energent.ai - AI-Powered Industry Analysis & Insights Applications Templates Resources Company Pricing Blog    │
│  Sign In English EN Sign In AI-Powered Industry Analysis Empower your team with automated market research,      │
│  competitive intelligence, and trend analysis, no code required. See Demo Get Started 4.9+/5 Product Rating     │
│  95% Client Satisfaction 3hrs Hours Saved Daily $80k Monthly Savings Trusted by teams at How It Works Watch     │
│  our AI agent autonomously analyze industry data and generate insightful visualizations, with full              │
│  transparency. Reviews Read what our customers are saying “ "We've tried numerous industry analysis tools, but  │
│  Energent.ai delivered the most accurate and comprehensive market insights from our reports." ” Richard Song    │
│  CEO-Epsilla “ "Energent.ai's advanced AI delivers where other platforms fail. Analyzing complex industry       │
│  reports and datasets requires this fusion of visual and textual understanding." ” Jon Conradt Principal        │
│  Scientist-AWS “ "It's far better than other tools! Our market analysts are able to triple their output,        │
│  generating deeper industry reports in a fraction of the time." ” Jamal CEO-xtrategise “ "Energent.ai           │
│  outperformed 10+ other platforms in our benchmarks, delivering top-tier accuracy in competitive analysis and   │
│  market trend identification." ” Ethan Zheng CTO - Jobright “ "As a business strategy consultant, I seek        │
│  state-of-the-art solutions for my clients. Energent.ai enhances the accuracy of our market intelligence... an  │
│  innovative tool for any analysis pipeline!" ” Cass Senior Scientist - AWS “ "I am impressed by Energent.ai's   │
│  innovation in the space of AI-driven industry analysis... and the power it gives our strategy teams." ” Felix  │
│  Bai Sr. Solution Architect - AWS “ "I have validated the quality of Energent.ai's analysis far beyond          │
│  traditional market research tools... Looking forward to using this in our future strategic planning." ” Steve  │
│  Cooper Cofounder - ai ticker chat “ "...                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Checker                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Report on Top Five Emerging AI Tools for Automating Competitive Market Analysis                            │
│                                                                                                                 │
│  #### 1. **Competely**                                                                                          │
│  - **Description**: AI-powered competitive analysis tool for startups, marketers, agencies, and consultants.    │
│  - **Key Features**: Offers automated report generation including SWOT analysis, real-time competitor           │
│  monitoring, actionable insights based on over 100 data points.                                                 │
│  - **Pricing**: $39/month (Standard).                                                                           │
│  - **Limitations**: Limited integration capabilities with other analytical tools.                               │
│  - **Ideal Use Cases**: Best suited for startups needing quick competitive insights.                            │
│  - **Source**: Verified from [G2](https://ai.g2.com/marketplace/tools/competely)                                │
│                                                                                                                 │
│  #### 2. **Klue**                                                                                               │
│  - **Description**: An AI platform designed for competitive intelligence and win-loss analysis providing        │
│  real-time data for sales teams.                                                                                │
│  - **Key Features**: Automated competitor tracking, personalized insights for sales, win-loss analysis.         │
│  - **Pricing**: Varies, direct consultation needed.                                                             │
│  - **Limitations**: Can be complex for smaller teams without dedicated resources.                               │
│  - **Ideal Use Cases**: Ideal for B2B organizations requiring comprehensive competitive insights.               │
│  - **Source**: Verified from [G2](https://ai.g2.com/marketplace/tools/klue)                                     │
│                                                                                                                 │
│  #### 3. **MarketGeist**                                                                                        │
│  - **Description**: AI-driven platform for continuous market and competitive analysis using 250+ data sources.  │
│  - **Key Features**: Real-time market signal monitoring, auto-generated insights, interactive analysis canvas.  │
│  - **Pricing**: Starting at $49/month (Starter plan).                                                           │
│  - **Limitations**: Requires some familiarity with AI tools; can have a learning curve.                         │
│  - **Ideal Use Cases**: Suitable for mid-sized marketing firms needing fast insights.                           │
│  - **Source**: Verified from [MarketGeist Website](https://marketgeist.com/)                                    │
│                                                                                                                 │
│  #### 4. **Cyndra**                                                                                             │
│  - **Description**: AI tool monitoring competitor moves

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query using all verified research data.          │
│  Structure it with clear sections, include citations, and provide actionable insights.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report on Top Five Emerging AI Tools for Automating Competitive Market Analysis                │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  In the rapidly evolving landscape of competitive market analysis, businesses increasingly leverage artificial  │
│  intelligence (AI) to gain insights and maintain a competitive edge. This report evaluates the top five         │
│  emerging AI tools specifically designed to automate competitive market analysis, highlighting their features,  │
│  limitations, cost analyses, and ideal use cases for mid-sized marketing firms. The findings consolidate        │
│  verified data from reputable sources to facilitate informed decision-making.                                   │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│                                                                                                                 │
│  ### 1. **Competely**                                                                                           │
│  - **Description**: An AI-powered competitive analysis tool that automates competitor tracking and analysis     │
│  for startups and mid-sized firms.                                                                              │
│  - **Key Features**:                                                                                            │
│    - Automated report generation including SWOT analysis and market positioning.                                │
│    - Real-time monitoring of competitors’ product updates and marketing strategies.                             │
│  - **Pricing**: $39/month (Standard).                                                                           │
│  - **Limitations**: Limited integration with other analytical tools could affect usability in more complex      │
│  environments.                                                                                                  │
│  - **Ideal Use Cases**: Suitable for startups seeking quick, actionable insights to enhance market              │
│  positioning.                                                                                                   │
│                                                                                                                 │
│  **Source**: Verified from [G2](https://ai.g2.com/marketplace/tools/competely)                                  │
│                                                                                                                 │
│  ### 2. **Klue**                                                                                                │
│  - **Description**: An AI platform designed for competitive intelligence, emphasizing win-loss analysis.        │
│  - **Key Features**:                                                                                            │
│    - Centralized dashboard for competitive data and insights.                                                   │
│    - Automated competitor tracking and personalized rec

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**📝 我的笔记 — 在 Jupyter 里跑 `crew.kickoff()` 可能会报 `RuntimeError`**

```
RuntimeError: Agent execution was invoked synchronously from within a running
event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()`
(or `await agent.aexecute_task(...)`) when calling from async code.
```

**原因**：Jupyter 的 `ipykernel` 内核本身一直跑着一个 asyncio 事件循环；这个版本的 `crewai` 内部走的是异步执行路径，检测到"当前已经处于一个运行中的事件循环"时就拒绝同步调用。纯 `.py` 脚本没有预先存在的事件循环，所以同样代码在脚本里能跑通，只有 notebook 里会炸。

**修复**：把上面这个 cell 改成异步调用（Jupyter 支持 cell 里直接 top-level `await`）：

```python
result = await crew.kickoff_async(
    inputs={"user_query": user_query}
)
```

或者保留同步写法，开头加一次 `nest_asyncio.apply()` 打补丁。

</div>


From the output of the previous cell check all the outputs for each task. Do they match what you expected? If not, go back and refine the `expected_output`. 

You can also print the final report to see the final result of the crew

In [23]:
from IPython.display import Markdown
Markdown(result.raw) 

# Comprehensive Report on Top Five Emerging AI Tools for Automating Competitive Market Analysis

## Executive Summary
In the rapidly evolving landscape of competitive market analysis, businesses increasingly leverage artificial intelligence (AI) to gain insights and maintain a competitive edge. This report evaluates the top five emerging AI tools specifically designed to automate competitive market analysis, highlighting their features, limitations, cost analyses, and ideal use cases for mid-sized marketing firms. The findings consolidate verified data from reputable sources to facilitate informed decision-making.

## Detailed Findings

### 1. **Competely**
- **Description**: An AI-powered competitive analysis tool that automates competitor tracking and analysis for startups and mid-sized firms.
- **Key Features**: 
  - Automated report generation including SWOT analysis and market positioning.
  - Real-time monitoring of competitors’ product updates and marketing strategies.
- **Pricing**: $39/month (Standard).
- **Limitations**: Limited integration with other analytical tools could affect usability in more complex environments.
- **Ideal Use Cases**: Suitable for startups seeking quick, actionable insights to enhance market positioning.
  
**Source**: Verified from [G2](https://ai.g2.com/marketplace/tools/competely)

### 2. **Klue**
- **Description**: An AI platform designed for competitive intelligence, emphasizing win-loss analysis.
- **Key Features**: 
  - Centralized dashboard for competitive data and insights.
  - Automated competitor tracking and personalized recommendations for sales teams.
- **Pricing**: Pricing varies. Direct consultation required.
- **Limitations**: Complexity of the system may be challenging for smaller teams without dedicated resources.
- **Ideal Use Cases**: Best for B2B organizations requiring comprehensive competitive insights to enhance sales strategies.
  
**Source**: Verified from [G2](https://ai.g2.com/marketplace/tools/klue)

### 3. **MarketGeist**
- **Description**: A continuous market and competitive analysis platform powered by AI with a focus on real-time data.
- **Key Features**: 
  - Real-time monitoring and analysis of market signals and competitor activities.
  - Reports generated from over 250 live data sources for comprehensive insights.
- **Pricing**: Starting at $49/month (Starter plan).
- **Limitations**: A steep learning curve may deter new users, making initial adoption variable.
- **Ideal Use Cases**: Effective for mid-sized marketing firms requiring fast, actionable insights; beneficial in dynamic markets.
  
**Source**: Verified from [MarketGeist](https://marketgeist.com/)

### 4. **Cyndra**
- **Description**: Focused on competitor move monitoring and market signals, Cyndra provides real-time analysis capabilities.
- **Key Features**: 
  - Automatic extraction of competitors’ changes, including product launches and pricing shifts.
  - User-friendly dashboard for ongoing competitor monitoring.
- **Pricing**: Starting at $99/month.
- **Limitations**: A primary focus on digital behavior might overlook offline competitors' impacts.
- **Ideal Use Cases**: Particularly useful for firms in fast-paced industries needing ongoing market trend insights.
  
**Source**: Verified from [Cyndra](https://www.cyndra.ai/use-cases/competitor-monitoring-intelligence)

### 5. **Energent.ai**
- **Description**: AI-driven platform for industry analysis and competitive intelligence.
- **Key Features**: 
  - Automates the market research process and provides advanced visualization of data.
  - Continuous updates and alerts based on ongoing data monitoring.
- **Pricing**: Custom pricing available; inquire for details.
- **Limitations**: The need for onboarding training can delay full capability utilization.
- **Ideal Use Cases**: Optimal for teams needing comprehensive data analysis while minimizing manual work.
  
**Source**: Verified from [Energent.ai](https://energent.ai/use-cases/en/industry-analysis)

## Comparative Analysis Table

| Tool          | Key Features                                        | Pricing         | Limitations                                          | Ideal Use Cases                                     |
|---------------|-----------------------------------------------------|------------------|-----------------------------------------------------|----------------------------------------------------|
| Competely     | Automated report generation, real-time monitoring   | $39/month        | Limited integrations                                | Startups needing quick insights                    |
| Klue          | Centralized data, personalized insights              | Variable         | Complexity for smaller teams                        | B2B organizations needing extensive insights        |
| MarketGeist   | Real-time market monitoring, extensive data sources  | Starting at $49/month | Steeper learning curve                              | Mid-sized firms looking for fast insights          |
| Cyndra        | Continuous monitoring, user-friendly dashboard       | Starting at $99/month | Primarily digital focus                              | Fast-paced industries wanting to track trends      |
| Energent.ai   | Automated insights, visual data representation      | Custom pricing    | Requires onboarding training                         | Teams seeking thorough analysis with less manual effort |

## Conclusion
This report thoroughly details the top five emerging AI tools for automating competitive market analysis, each characterized by unique features and limitations tailored to different business scenarios. By understanding the functionalities and pricing of these tools, mid-sized marketing firms can strategically select the most appropriate solutions to enhance their market analysis processes. These insights foster better decision-making, optimizing marketing strategies to align with dynamic market conditions.

### References
1. Competely. (n.d.). Retrieved from [G2](https://ai.g2.com/marketplace/tools/competely)
2. Klue. (n.d.). Retrieved from [G2](https://ai.g2.com/marketplace/tools/klue)
3. MarketGeist. (n.d.). Retrieved from [MarketGeist](https://marketgeist.com/)
4. Cyndra. (n.d.). Retrieved from [Cyndra](https://www.cyndra.ai/use-cases/competitor-monitoring-intelligence)
5. Energent.ai. (n.d.). Retrieved from [Energent.ai](https://energent.ai/use-cases/en/industry-analysis)

You made it to the end of the lab! You can go back and experiment with the goals and backstories of the agents, as well as description and expected outputs of tasks. You can also change the inputs to any research topic you wish. Have fun with it!